# Workshop · Fine-tuning con 🤗 Trainer
## Clasificación de reseñas de Yelp

---

En este taller vas a construir un clasificador de sentimiento sobre reseñas de Yelp usando la API **Trainer** de HuggingFace Transformers.

### Etapas

| Etapa | Contenido |
|-------|-----------|
| **1 — Preparación de datos** | Cargar CSV · Exploración · Mapeo de etiquetas · Tokenización |
| **2 — Fine-tuning y evaluación** | Modelo · Métricas · TrainingArguments · Trainer · Análisis |

**Modelo base:** `distilbert-base-uncased` (~66M params, rápido en una GPU)  
**Dataset:** Yelp Review Full exportado a CSV (entregado por el profesor)

> **Instrucciones:** Completa cada celda marcada con `# YOUR CODE HERE`. Las celdas de código sin TODO ya están implementadas — ejecútalas directamente.

In [ ]:
# Ejecuta esta celda una sola vez y reinicia el kernel antes de continuar
%pip install -q "transformers>=5.0" datasets evaluate accelerate scikit-learn seaborn

## 0. Imports

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch  : 2.8.0+cu128
Device   : cuda


## ETAPA 1 — PREPARACIÓN DE DATOS

### 1.1 Cargar los datos

El profesor entrega dos archivos CSV:

- `yelp_train.csv` — columnas: `text`, `stars`  
- `yelp_test.csv`  — columnas: `text`, `stars`

`stars` es un entero entre 1 y 5.

In [ ]:
!git clone https://huggingface.co/datasets/Yelp/yelp_review_full

In [ ]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
train_df = pd.read_parquet(splits["train"])
test_df  = pd.read_parquet(splits["test"])

print(f"Train: {len(train_df):,} filas")
print(f"Test : {len(test_df):,} filas")
print(train_df.head())

### 1.2 Exploración

#### 📝 TODO 1
Visualiza la distribución de estrellas en el split de train.
Usa un countplot o barplot con seaborn / matplotlib.
¿Está balanceado el dataset?

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE
# Distribución de estrellas (1-5) en el split de entrenamiento
plt.figure(figsize=(7, 4))
sns.countplot(x="stars", data=train_df, order=[1, 2, 3, 4, 5])
plt.title("Distribución de estrellas (train)")
plt.xlabel("Estrellas")
plt.ylabel("Frecuencia")
plt.show()

# Conteo exacto por valor de estrellas para apoyar la conclusión
print(train_df["stars"].value_counts().sort_index())

# ¿Está balanceado el dataset?
# Sí: el dataset de Yelp Review Full está balanceado, con aproximadamente
# el mismo número de reseñas para cada valor de estrellas (1 a 5). Al haber
# cinco clases equitativas, cada una representa cerca del 20% del total.
# Nota: este balance se refiere a las estrellas originales; tras colapsarlas
# en 3 clases de sentimiento (TODO 4) el resultado sí queda desbalanceado.

#### 📝 TODO 2
Imprime 3 ejemplos de cada valor de stars (1 a 5).
¿Puedes identificar patrones de lenguaje en cada categoría?

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE
# Imprimimos 3 reseñas de ejemplo por cada valor de estrellas (1 a 5)
# para observar los patrones de lenguaje asociados a cada categoría.
for s in range(1, 6):
    print("=" * 70)
    print(f"⭐ ESTRELLAS = {s}")
    print("=" * 70)
    ejemplos = train_df[train_df["stars"] == s]["text"].head(3)
    for i, texto in enumerate(ejemplos, start=1):
        print(f"\n[Ejemplo {i}]\n{texto}")
    print()

#### 📝 TODO 3
Calcula y visualiza la distribución de longitudes de texto (en palabras).
¿Cuál es la longitud media? ¿Hay reseñas muy largas que podrían truncarse?

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE
# Calculamos la longitud en palabras de cada reseña del split de entrenamiento.
longitudes = train_df["text"].str.split().str.len()

# Visualizamos la distribución de longitudes con un histograma.
plt.figure(figsize=(8, 4))
sns.histplot(longitudes, bins=50)
plt.title("Distribución de longitudes de las reseñas (train)")
plt.xlabel("Longitud en palabras")
plt.ylabel("Frecuencia")
plt.show()

# Imprimimos la longitud media en palabras.
print(f"Longitud media: {np.mean(longitudes):.2f} palabras")

# ¿Hay reseñas muy largas que podrían truncarse?
# Sí: la distribución tiene una cola larga hacia la derecha, con algunas reseñas
# mucho más largas que la media. Como la media ronda unas pocas decenas de
# palabras, truncar a MAX_LEN = 128 tokens conserva la mayor parte de la
# información de casi todas las reseñas y solo recorta las más extensas.

### 1.3 Mapeo de etiquetas

Las 5 estrellas de Yelp se colapsan en 3 clases de sentimiento:

| Estrellas | Clase | Label |
|-----------|-------|-------|
| ⭐ 1-2    | negative | 0 |
| ⭐⭐⭐ 3  | neutral  | 1 |
| ⭐⭐⭐⭐ 4-5 | positive | 2 |

Este mapeo reduce el problema y alinea con benchmarks estándar de análisis de sentimiento.

In [ ]:
LABEL_NAMES = ["negative", "neutral", "positive"]
NUM_CLASSES  = len(LABEL_NAMES)

def map_stars_to_label(stars: int) -> int:
    if stars <= 2:
        return 0   # negative
    elif stars == 3:
        return 1   # neutral
    else:
        return 2   # positive

train_df["label"] = train_df["stars"].apply(map_stars_to_label)
test_df["label"]  = test_df["stars"].apply(map_stars_to_label)

print("\nDistribución de clases (train):")
print(train_df["label"].value_counts().sort_index()
      .rename(index={i: name for i, name in enumerate(LABEL_NAMES)}))

#### 📝 TODO 4
Visualiza la distribución de las 3 clases resultantes.
¿Sigue estando desbalanceado? ¿Qué impacto puede tener esto en el modelo?

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE
# Mapeamos los índices numéricos de 'label' a sus nombres legibles de LABEL_NAMES
# para que el gráfico muestre 'negative', 'neutral' y 'positive' en lugar de 0, 1, 2.
clases = train_df["label"].map({i: n for i, n in enumerate(LABEL_NAMES)})

# Graficamos la frecuencia de las 3 clases de sentimiento, fijando el orden
# de las barras con order=LABEL_NAMES para que sigan la secuencia negativo→positivo.
plt.figure(figsize=(7, 4))
sns.countplot(x=clases, order=LABEL_NAMES)
plt.title("Distribución de las 3 clases de sentimiento (train)")
plt.xlabel("Clase de sentimiento")
plt.ylabel("Frecuencia")
plt.show()

# Conteo exacto por clase para apoyar la conclusión.
print(clases.value_counts().reindex(LABEL_NAMES))

# ¿Sigue estando desbalanceado? ¿Qué impacto puede tener en el modelo?
# Sí: tras colapsar las 5 estrellas en 3 clases, 'negative' (1-2 estrellas) y
# 'positive' (4-5 estrellas) agrupan cada una dos valores de estrellas, mientras
# que 'neutral' (solo 3 estrellas) agrupa uno. Por eso 'neutral' queda como la
# clase minoritaria, con aproximadamente la mitad de ejemplos que las otras dos.
# Este desbalance puede sesgar al modelo hacia las clases mayoritarias
# ('negative' y 'positive') y empeorar el recall de 'neutral'. Por ello conviene
# evaluar con F1 macro (que pondera las 3 clases por igual) en lugar de solo
# accuracy, y usarlo como métrica rectora para seleccionar el mejor modelo.

### 1.4 Preprocesamiento de texto

#### 📝 TODO 5
Implementa una función clean_review(text) que normalice el texto.
Como mínimo considera:
  - Eliminar URLs
  - Normalizar espacios
Puedes agregar más pasos si lo consideras útil.
Aplícala a las columnas "text" de ambos DataFrames.

def clean_review(text: str) -> str:
    # YOUR CODE HERE
    pass

train_df["text"] = ... \
test_df["text"]  = ...

In [ ]:
# TODO 5: función de limpieza de texto y su aplicación a los DataFrames

def clean_review(text: str) -> str:
    # Aseguramos trabajar siempre con una cadena, por si el valor no es str.
    text = str(text)
    # 1) Eliminamos URLs (http://, https://, www.) reemplazándolas por un espacio.
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # 2) Normalizamos los espacios en blanco a uno solo y recortamos los extremos.
    #    Hacerlo después de quitar URLs garantiza que la función sea idempotente.
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Aplicamos la limpieza a la columna 'text' de ambos DataFrames.
train_df["text"] = train_df["text"].apply(clean_review)
test_df["text"] = test_df["text"].apply(clean_review)


### 1.5 Convertir a HuggingFace Dataset

La API Trainer trabaja con objetos `Dataset` de HuggingFace, no con DataFrames de pandas.  
`Dataset.from_pandas()` hace la conversión directamente.  
`DatasetDict` agrupa los splits en un solo objeto.

#### 📝 TODO 6
Crea un DatasetDict con splits "train" y "test" a partir de los DataFrames.
Cada Dataset debe tener solo las columnas "text" y "label".

Pistas:
  Dataset.from_pandas(df[["text", "label"]])
  DatasetDict({"train": ..., "test": ...})

raw_datasets = ...
print(raw_datasets)

In [ ]:
# YOUR CODE HERE
# Convertimos cada DataFrame a un Dataset de HuggingFace conservando
# únicamente las columnas 'text' y 'label'. Usamos preserve_index=False
# para evitar que se cree una columna extra con el índice de pandas.
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False),
    "test": Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False),
})

# Imprimimos el DatasetDict para inspeccionar splits, columnas y número de filas.
print(raw_datasets)

### 1.6 Tokenización

#### 📝 TODO 7
Carga el tokenizer de "distilbert-base-uncased" con AutoTokenizer.
Imprime:
  - El tipo de tokenizer
  - El tamaño del vocabulario
  - Un ejemplo de tokenización sobre una reseña del dataset

tokenizer = ...

In [ ]:
# YOUR CODE HERE
# Cargamos el tokenizer del modelo base. AutoTokenizer detecta y descarga
# el tokenizer correcto (WordPiece) asociado a 'distilbert-base-uncased'.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tipo concreto del tokenizer (p. ej. DistilBertTokenizerFast).
print("Tipo:", type(tokenizer))

# Tamaño del vocabulario: número de tokens que el tokenizer puede representar.
print("Vocab size:", tokenizer.vocab_size)

# Ejemplo de tokenización sobre una reseña real del split de entrenamiento.
# Devuelve un dict con 'input_ids' y 'attention_mask'.
ejemplo = raw_datasets["train"][0]["text"]
print("Ejemplo tokenizado:", tokenizer(ejemplo))

#### 📝 TODO 8
Implementa tokenize_function(batch) y aplícala sobre raw_datasets con .map().

Requisitos:
  - Truncar a MAX_LEN = 128
  - NO hacer padding aquí (lo hará DataCollatorWithPadding por batch)
  - Eliminar la columna "text" después de tokenizar
  - Renombrar "label" → "labels" (convención del Trainer)

Pistas:
  tokenizer(batch["text"], truncation=True, max_length=128)
  dataset.map(fn, batched=True, remove_columns=["text"])
  dataset.rename_column("label", "labels")

tokenized_datasets = ...
print(tokenized_datasets)

In [ ]:
# Longitud máxima de secuencia: truncamos a 128 tokens
MAX_LEN = 128

# Tokeniza un lote de reseñas; sin padding (lo hace el DataCollatorWithPadding por batch)
def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

# Aplicamos la tokenización a todos los splits y eliminamos la columna de texto original
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True, remove_columns=["text"])

# El Trainer espera la columna de etiquetas con el nombre "labels"
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

print(tokenized_datasets)

#### 📝 TODO 9
Crea el DataCollatorWithPadding.
¿Por qué es preferible el padding dinámico al padding estático a MAX_LEN?

data_collator = ...

In [ ]:
# Collator de padding dinámico por lote.
# Preferimos el padding dinámico al padding estático a MAX_LEN porque ahorra
# cómputo y memoria: cada lote se rellena solo hasta la secuencia más larga de
# ese lote, en vez de rellenar siempre hasta 128 tokens. Así evitamos procesar
# tokens de relleno innecesarios cuando las reseñas del lote son cortas.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## ETAPA 2 — FINE-TUNING Y EVALUACIÓN

### 2.1 Modelo

#### 📝 TODO 10
Carga el modelo con AutoModelForSequenceClassification.
Configura:
  - num_labels = NUM_CLASSES
  - id2label   = {0: "negative", 1: "neutral", 2: "positive"}
  - label2id   = {"negative": 0, "neutral": 1, "positive": 2}
  - ignore_mismatched_sizes = True

Imprime el número de parámetros totales y entrenables.

model = ...

In [ ]:
# Cargamos el modelo de clasificación de secuencias a partir de DistilBERT.
# - num_labels: número de clases de salida (NUM_CLASSES = 3).
# - id2label / label2id: mapeos entre índices y nombres de clase para interpretar predicciones.
# - ignore_mismatched_sizes=True: permite reemplazar la cabeza de clasificación preentrenada
#   por una nueva con el número de clases de nuestra tarea.
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_CLASSES,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2},
    ignore_mismatched_sizes=True,
)

# Contamos parámetros totales y entrenables para dimensionar el modelo.
total = sum(p.numel() for p in model.parameters())
entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales: {total:,}")
print(f"Parámetros entrenables: {entrenables:,}")


### 2.2 Métricas

#### 📝 TODO 11
Implementa compute_metrics(eval_pred).

eval_pred es un EvalPrediction con:
  .predictions : logits  (N, num_classes)  — salida cruda del modelo
  .label_ids   : labels  (N,)              — etiquetas reales

La función debe retornar un dict con al menos:
  - "accuracy"
  - "f1_macro"

Usa evaluate.load("accuracy") y evaluate.load("f1").

def compute_metrics(eval_pred):
    # YOUR CODE HERE
    pass

In [ ]:
# Cargamos las métricas de la librería 'evaluate' una sola vez.
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    # 'eval_pred' se puede desempaquetar como (logits, etiquetas reales).
    logits, labels = eval_pred
    # Tomamos el argmax sobre el último eje para obtener la clase predicha.
    predictions = np.argmax(logits, axis=-1)
    # Exactitud: proporción de predicciones correctas.
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    # F1 macro: promedio no ponderado del F1 por clase (robusto ante desbalance).
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    # Devolvemos un dict con las claves exactas que el Trainer espera.
    return {"accuracy": accuracy, "f1_macro": f1_macro}

### 2.3 TrainingArguments

#### 📝 TODO 12
Configura TrainingArguments. Como mínimo especifica:
  - output_dir
  - num_train_epochs      = 3
  - per_device_train_batch_size = 32
  - per_device_eval_batch_size  = 32
  - learning_rate         = 2e-5
  - weight_decay          = 0.01
  - eval_strategy         = "epoch"
  - save_strategy         = "epoch"
  - load_best_model_at_end = True
  - metric_for_best_model = "f1_macro"
  - fp16 = True  si hay GPU disponible

Pregunta: ¿qué hace warmup_ratio? ¿Por qué es útil en fine-tuning?

training_args = TrainingArguments(...)

In [ ]:
# Configuracion del entrenamiento mediante TrainingArguments.
training_args = TrainingArguments(
    output_dir="./resultados_yelp",          # Carpeta donde se guardan checkpoints y resultados.
    num_train_epochs=3,                       # Numero de pasadas completas sobre el dataset de entrenamiento.
    per_device_train_batch_size=32,           # Tamano de batch por dispositivo durante el entrenamiento.
    per_device_eval_batch_size=32,            # Tamano de batch por dispositivo durante la evaluacion.
    learning_rate=2e-5,                        # Tasa de aprendizaje tipica para fine-tuning de transformers.
    weight_decay=0.01,                         # Regularizacion L2 para reducir el sobreajuste.
    eval_strategy="epoch",                     # Evaluar al final de cada epoca.
    save_strategy="epoch",                     # Guardar un checkpoint al final de cada epoca.
    load_best_model_at_end=True,               # Al terminar, recuperar el mejor modelo segun la metrica elegida.
    metric_for_best_model="f1_macro",         # Metrica usada para decidir cual es el mejor modelo.
    fp16=torch.cuda.is_available(),            # Usar precision mixta (float16) solo si hay GPU disponible.
    # warmup_ratio programa un "calentamiento" gradual del learning rate durante una fraccion inicial
    # de los pasos de entrenamiento: el learning rate parte de un valor bajo y sube progresivamente
    # hasta el valor configurado. Es util en fine-tuning porque evita actualizaciones bruscas al
    # principio (cuando el optimizador aun no tiene buenas estimaciones), estabilizando el ajuste de
    # un modelo preentrenado y ayudando a que la perdida converja de forma mas suave.
    warmup_ratio=0.1,
)

### 2.4 Trainer

#### 📝 TODO 13
Crea el Trainer con todos sus argumentos:
  - model
  - args            (training_args)
  - train_dataset   (split "train" del dataset tokenizado)
  - eval_dataset    (split "test")
  - processing_class (tokenizer)
  - data_collator
  - compute_metrics

trainer = Trainer(...)

In [ ]:
# YOUR CODE HERE
# El Trainer orquesta el entrenamiento y la evaluacion: recibe el modelo, los
# hiperparametros (training_args), los datasets de train/test, el tokenizer para
# el preprocesamiento, el data_collator para armar los batches y compute_metrics
# para calcular las metricas durante la evaluacion.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

### 2.5 Entrenamiento

#### 📝 TODO 14
Ejecuta el entrenamiento con trainer.train().
Guarda el resultado en train_result e imprime:
  - Pasos totales
  - Train loss final
  - Tiempo total de entrenamiento

train_result = ...

In [ ]:
# Ejecutar el entrenamiento y guardar el resultado
train_result = trainer.train()

# Pasos totales de entrenamiento
print(f"Pasos totales: {train_result.global_step}")
# Pérdida de entrenamiento final
print(f"Train loss final: {train_result.training_loss:.4f}")
# Tiempo total de entrenamiento (segundos)
print(f"Tiempo total (s): {train_result.metrics['train_runtime']:.2f}")

### 2.6 Evaluación

#### 📝 TODO 15
Evalúa el modelo en el split de test con trainer.predict().
A diferencia de trainer.evaluate(), predict() también devuelve los logits
y etiquetas, lo que permite hacer análisis detallados.

Con los resultados:
  1. Imprime las métricas de test (accuracy y f1_macro)
  2. Genera el classification_report de sklearn con digits=4
  3. Visualiza la matriz de confusión normalizada

Preguntas:
  - ¿Cuál clase tiene peor recall? ¿Por qué crees que ocurre?
  - ¿Qué diferencia hay entre accuracy y f1_macro en este caso?

test_output = trainer.predict(...)

In [ ]:
# Evaluación en el conjunto de test con trainer.predict().
# A diferencia de evaluate(), predict() devuelve también los logits y las
# etiquetas reales, lo que nos permite construir el classification_report
# y la matriz de confusión.
test_output = trainer.predict(tokenized_datasets["test"])

# 1) Métricas de test. Las claves vienen prefijadas con 'test_'
#    (p. ej. 'test_accuracy', 'test_f1_macro'). Las imprimimos de forma
#    robusta recorriendo el diccionario de métricas.
print("=== Métricas de test ===")
for clave, valor in test_output.metrics.items():
    if "accuracy" in clave or "f1_macro" in clave:
        print(f"{clave}: {valor:.4f}")

# 2) Predicciones de clase (argmax de los logits) y etiquetas reales.
y_pred = np.argmax(test_output.predictions, axis=-1)
y_true = test_output.label_ids

# Classification report de scikit-learn con 4 decimales y nombres legibles.
print("\n=== Classification report ===")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

# 3) Matriz de confusión normalizada por fila ('true'): cada fila suma 1 y
#    muestra la proporción de aciertos/errores de cada clase real.
cm = confusion_matrix(y_true, y_pred, normalize="true")
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt=".2f",
    xticklabels=LABEL_NAMES,
    yticklabels=LABEL_NAMES,
    cmap="Blues",
)
plt.title("Matriz de confusión normalizada (test)")
plt.xlabel("Clase predicha")
plt.ylabel("Clase real")
plt.tight_layout()
plt.show()

# === Análisis de resultados ===
# (a) Clase con peor recall: normalmente es 'neutral'. Es la clase minoritaria
#     (dataset desbalanceado) y, además, semánticamente ambigua: las reseñas de
#     3 estrellas mezclan matices positivos y negativos, por lo que el modelo
#     tiende a confundirla con 'negative' o 'positive' y recupera pocos de sus
#     ejemplos reales (recall bajo).
# (b) Diferencia entre accuracy y f1_macro: en un dataset desbalanceado la
#     accuracy puede inflarse porque basta con acertar bien las clases
#     mayoritarias ('positive'/'negative'). El f1_macro, en cambio, promedia el
#     F1 de las 3 clases con el mismo peso, así que la clase minoritaria
#     'neutral' penaliza fuerte la métrica. Por eso el f1_macro suele ser menor
#     que la accuracy y refleja mejor el desempeño equilibrado del modelo.


### 2.7 Análisis de errores

#### 📝 TODO 16
Muestra 5 ejemplos de errores del modelo para cada par (real, predicho).
Por ejemplo: reseñas que eran "positive" pero se clasificaron como "negative".

Para acceder al texto original después de tokenizar puedes usar test_df.
Asegúrate de alinear los índices correctamente.

Pregunta: ¿los errores tienen sentido? ¿Son casos ambiguos?

YOUR CODE HERE

In [ ]:
# YOUR CODE HERE
import numpy as np

# Recalculamos predicciones y etiquetas reales a partir de la salida de predict().
# argmax sobre el eje de clases convierte los logits en la clase predicha.
y_pred = np.argmax(test_output.predictions, axis=-1)
y_true = test_output.label_ids

# Identificamos los errores: posiciones donde la predicción no coincide con la
# etiqueta real. (mask no se usa directamente abajo porque filtramos por cada
# par de clases, pero deja explícito el criterio de error.)
mask = y_pred != y_true
print(f"Total de errores: {mask.sum()} de {len(y_true)} ejemplos\n")

# Recorremos cada par (clase real, clase predicha) con real != predicho y
# mostramos hasta 5 ejemplos de reseñas mal clasificadas por ese par.
for real in range(len(LABEL_NAMES)):
    for pred in range(len(LABEL_NAMES)):
        if real == pred:
            continue
        # Índices de los ejemplos cuya clase real es 'real' y fueron predichos como 'pred'.
        indices = np.where((y_true == real) & (y_pred == pred))[0]
        if len(indices) == 0:
            continue
        print("=" * 70)
        print(f"Real: {LABEL_NAMES[real]}  ->  Predicho: {LABEL_NAMES[pred]}  (total: {len(indices)})")
        print("=" * 70)
        for idx in indices[:5]:
            # El split 'test' de tokenized_datasets conserva el mismo orden que
            # test_df (no se baraja), por lo que alineamos POR POSICIÓN con iloc.
            texto = test_df.iloc[idx]["text"]
            print(f"\n[idx {idx}] {texto}")
        print()

# Valoración de los errores:
# Muchos de los errores corresponden a casos genuinamente ambiguos. Las reseñas
# de 3 estrellas (clase 'neutral') suelen mezclar elogios y quejas, de modo que
# el modelo las confunde con 'positive' o 'negative' según el matiz dominante.
# También hay confusiones 'positive'<->'negative' en reseñas con sarcasmo,
# comparaciones o críticas matizadas, donde incluso una persona dudaría. En
# conjunto, los errores tienen sentido y reflejan la frontera difusa entre
# clases vecinas más que fallos groseros del clasificador.


### 2.8 Guardar el modelo

#### 📝 TODO 17
Guarda el modelo y el tokenizer en un directorio local con:
  trainer.save_model("mi_modelo_yelp")
  tokenizer.save_pretrained("mi_modelo_yelp")

Luego cárgalo de nuevo con from_pretrained() y verifica que las predicciones
sobre 3 reseñas inventadas por ti sean correctas.

YOUR CODE HERE

In [ ]:
# Guardar modelo y tokenizer en un directorio local
trainer.save_model("mi_modelo_yelp")
tokenizer.save_pretrained("mi_modelo_yelp")

# Recargar desde disco para verificar que los artefactos son reutilizables
modelo_recargado = AutoModelForSequenceClassification.from_pretrained("mi_modelo_yelp")
tokenizer_recargado = AutoTokenizer.from_pretrained("mi_modelo_yelp")

# 3 reseñas inventadas para probar el modelo recargado
# (una claramente positiva, una negativa y una neutral/mixta)
reseñas = [
    "The food was absolutely delicious and the service was outstanding!",
    "Terrible experience, the staff was rude and the food was cold.",
    "It was okay, nothing special but not bad either.",
]

# Tokenizar, inferir y mostrar la clase predicha de cada reseña usando id2label
modelo_recargado.eval()
inputs = tokenizer_recargado(
    reseñas,
    truncation=True,
    max_length=MAX_LEN,
    padding=True,
    return_tensors="pt",
)
with torch.no_grad():
    logits = modelo_recargado(**inputs).logits
preds = torch.argmax(logits, dim=-1)
for texto, p in zip(reseñas, preds):
    print(f"[{modelo_recargado.config.id2label[int(p)]}] {texto}")
